# Chạy lại hai lượt dừng sớm — yolo26s-obb vòng 0, hạt 1 và 2, patience 50

Gói `colab_B` đã chạy xong 24/24 lượt (21/09). Trong Bảng A, `yolo26s-obb` vòng 0 có hạt 1 dừng ở epoch 56 và hạt 2 dừng ở epoch 43 (patience 20 trên val 201 khung), còn hạt 0 chạy 130 epoch. Hai lượt dừng sớm là hai lượt thấp nhất bảng (mAP50 3 lớp 0,809 / 0,812 so với 0,865 của hạt 0). Notebook này chạy lại đúng hai lượt đó với patience 50.

**Không đụng kết quả cũ:** JSON mới ghi tên `yolo26s-obb_v0_s1_p50.json`, `yolo26s-obb_v0_s2_p50.json`; runs mới ở `ket_qua/runs/yolo26s-obb_v0_s1_p50/`. 24 JSON cũ và `chay_bangA_B3.ipynb` giữ nguyên.

**Chạy:** ô 1 → 3 (mount, giải nén, hàm), rồi ô 4 (hai lượt, mỗi lượt 1–2 giờ T4), rồi ô 5 xem bảng. Đứt phiên: chạy lại từ ô 1, lượt có JSON tự bỏ qua, lượt dở nối từ `last.pt`.

**Đọc:** hai lượt lên quanh 0,86 ⇒ bốn kiến trúc bằng nhau, giữ yolo26s-obb. Vẫn quanh 0,81 ⇒ dừng sớm không phải nguyên nhân.


In [2]:
# 1. Drive + giải nén
from google.colab import drive; drive.mount('/content/drive')
import os, zipfile, json, csv, time, glob, shutil
D = '/content/drive/MyDrive/colab_B'
assert os.path.isdir(D), D
KQ = f'{D}/ket_qua'; os.makedirs(KQ, exist_ok=True)
if not os.path.isdir('/content/khung/images'):
    t = time.time(); zipfile.ZipFile(f'{D}/khung_nhan.zip').extractall('/content/khung'); print('giải nén %.0f s' % (time.time() - t))
rows = list(csv.DictReader(open(f'{D}/chia_khoi.csv', encoding='utf8')))
print(len(rows), 'khung |', len(os.listdir('/content/khung/images')), 'ảnh |', len(os.listdir('/content/khung/labels')), 'nhãn')
!pip -q install ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
giải nén 13 s
3000 khung | 3000 ảnh | 3000 nhãn
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 10.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ultralytics 8.4.157
Tesla T4, 15360 MiB


In [3]:
# 2. Dựng thư mục theo vai từng vòng (symlink, không chép ảnh) + data.yaml cho OBB và hộp thẳng
LOP = ['SingleHotSpot', 'MultiHotSpot', 'Bypass', 'ModuleOpenCircuit']
K = 5

def obb_sang_hop_thang(src, dst):
    # 8 số 4 góc (chuẩn hoá) -> cx cy w h
    out = []
    for l in open(src, encoding='utf8').read().splitlines():
        q = l.split()
        if len(q) != 9: continue
        xs = [float(q[k]) for k in (1, 3, 5, 7)]; ys = [float(q[k]) for k in (2, 4, 6, 8)]
        x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
        out.append('%s %.6f %.6f %.6f %.6f' % (q[0], (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
    open(dst, 'w').write('\n'.join(out) + ('\n' if out else ''))

def dung(vong, kieu):
    # kieu: 'obb' | 'hbb'
    goc = f'/content/ds_{kieu}_v{vong}'
    if os.path.isdir(goc): return goc
    for vai in ('train', 'val', 'test'):
        os.makedirs(f'{goc}/{vai}/images'); os.makedirs(f'{goc}/{vai}/labels')
    for r in rows:
        vai = r[f'vai_v{vong}']
        if vai == 'dem': continue            # đệm: bỏ hẳn
        ten = r['file'][:-4]
        os.symlink(f'/content/khung/images/{r["file"]}', f'{goc}/{vai}/images/{r["file"]}')
        src = f'/content/khung/labels/{ten}.txt'
        dst = f'{goc}/{vai}/labels/{ten}.txt'
        if kieu == 'obb': os.symlink(src, dst)
        else: obb_sang_hop_thang(src, dst)
    open(f'{goc}/data.yaml', 'w').write(
        f'path: {goc}\ntrain: train/images\nval: val/images\ntest: test/images\nnc: 4\nnames: {LOP}\n')
    return goc

for v in range(K):
    for kieu in ('obb', 'hbb'):
        g = dung(v, kieu)
    n = {vai: len(os.listdir(f'{g}/{vai}/images')) for vai in ('train', 'val', 'test')}
    print(f'vòng {v}: {n}')

vòng 0: {'train': 2111, 'val': 201, 'test': 498}
vòng 1: {'train': 2241, 'val': 221, 'test': 294}
vòng 2: {'train': 1990, 'val': 203, 'test': 474}
vòng 3: {'train': 2176, 'val': 215, 'test': 354}
vòng 4: {'train': 1806, 'val': 162, 'test': 885}


In [4]:
# 3. Hàm train + chấm một lượt. Ghi JSON ngay khi xong; có JSON thì bỏ qua.
from ultralytics import YOLO
import numpy as np, torch

CAU_HINH = dict(epochs=150, patience=20, batch=16, imgsz=640, lr0=0.005, degrees=15,
                flipud=0.5, fliplr=0.5, mosaic=1.0, close_mosaic=15, optimizer='auto',
                workers=4, plots=False, verbose=False)
KIEN_TRUC = {           # tên: (yaml, kiểu hộp)
    'yolo26s-obb': ('yolo26s-obb.yaml', 'obb'),
    'yolov9c':     ('yolov9c.yaml',     'hbb'),
    'yolo26s':     ('yolo26s.yaml',     'hbb'),
    'yolov8s-obb': ('yolov8s-obb.yaml', 'obb'),
}

def theo_lop(b, thuoc):
    # b.p / b.r / b.ap50 chỉ có ở lớp CÓ mẫu trong test; ánh xạ về đủ 4 lớp qua ap_class_index, lớp vắng = None
    v = getattr(b, thuoc); ra = [None] * 4
    for k, ci in enumerate(b.ap_class_index):
        ra[int(ci)] = round(float(v[k]), 4)
    return ra

def cham_tai_conf(model, data_yaml, conf):
    # OBBMetrics lẫn DetMetrics đều để kết quả ở .box (ultralytics 8.4)
    r = model.val(data=data_yaml, split='test', conf=conf, iou=0.5, plots=False, verbose=False)
    b = r.box
    return dict(P_theo_lop=theo_lop(b, 'p'), R_theo_lop=theo_lop(b, 'r'), mAP50=round(float(b.map50), 4))

def mot_luot(ten, vong, hat, bien_the='', **de):
    # bien_the: đuôi tag cho lượt chạy lại (vd 'p50'); de: cấu hình đè lên CAU_HINH (vd patience=50)
    yaml_, kieu = KIEN_TRUC[ten]
    ch = dict(CAU_HINH, **de)
    tag = f'{ten}_v{vong}_s{hat}' + (f'_{bien_the}' if bien_the else '')
    ra = f'{KQ}/{tag}.json'
    if os.path.exists(ra):
        print('  bỏ qua, đã có:', tag); return json.load(open(ra))
    goc = dung(vong, kieu); dy = f'{goc}/data.yaml'
    run = f'{KQ}/runs/{tag}'
    last = f'{run}/weights/last.pt'
    t0 = time.time()
    if os.path.exists(last):
        print('  nối tiếp từ last.pt:', tag)
        m = YOLO(last); m.train(resume=True)
    else:
        m = YOLO(yaml_)
        m.train(data=dy, seed=hat, project=f'{KQ}/runs', name=tag, exist_ok=True, **ch)
    best = f'{run}/weights/best.pt'
    m = YOLO(best)
    # chấm test: mAP (conf mặc định 0,001) + P/R ở conf 0,25 và 0,5
    r = m.val(data=dy, split='test', plots=False, verbose=False)
    b = r.box
    kq = dict(kien_truc=ten, yaml=yaml_, kieu_hop=kieu, vong=vong, hat=hat, bien_the=bien_the,
              tham_so_M=round(sum(p.numel() for p in m.model.parameters()) / 1e6, 2),
              epoch_chay=int(len(open(f'{run}/results.csv').read().splitlines()) - 1),
              phut=round((time.time() - t0) / 60, 1),
              test=dict(mAP50=round(float(b.map50), 4), mAP50_95=round(float(b.map), 4),
                        AP50_theo_lop=theo_lop(b, 'ap50'), lop=LOP),
              conf_0_25=cham_tai_conf(m, dy, 0.25),
              conf_0_5=cham_tai_conf(m, dy, 0.5),
              cau_hinh=ch)
    json.dump(kq, open(ra, 'w'), ensure_ascii=False, indent=1)
    print(f'  {tag}: mAP50 {kq["test"]["mAP50"]:.4f}  mAP50-95 {kq["test"]["mAP50_95"]:.4f}  '
          f'AP50 theo lớp {kq["test"]["AP50_theo_lop"]}  ({kq["phut"]} phút, {kq["epoch_chay"]} epoch)')
    return kq   # giữ best.pt trong runs/ để chấm lại sau

In [ ]:
# 4. CHẠY LẠI hai lượt dừng sớm của yolo26s-obb vòng 0 (hạt 1 dừng ở epoch 56, hạt 2 ở epoch 43; hạt 0 chạy 130).
# patience 20 trên val 201 khung cắt quá sớm. Chạy lại với patience 50, ghi JSON riêng "_p50", KHÔNG đè lượt cũ.
# Đọc kết quả: nếu hai lượt này lên ~0,86 (mAP50 3 lớp) thì Bảng A bốn kiến trúc bằng nhau; nếu vẫn ~0,81 thì mới là chuyện kiến trúc.
LICH_LAI = [('yolo26s-obb', 0, 1), ('yolo26s-obb', 0, 2)]
for ten, v, s in LICH_LAI:
    print(f'=== {ten}  vòng {v}  hạt {s}  patience 50 ===')
    mot_luot(ten, v, s, bien_the='p50', patience=50)


=== yolo26s-obb  vòng 0  hạt 1  patience 50 ===
  nối tiếp từ last.pt: yolo26s-obb_v0_s1_p50
Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ds_obb_v0/data.yaml, degrees=15, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/My

In [ ]:
# 5. Gom bảng — chạy bất cứ lúc nào, đọc mọi JSON đang có
import pandas as pd
fs = sorted(glob.glob(f'{KQ}/*.json'))
R = [json.load(open(f)) for f in fs]
df = pd.DataFrame([dict(kien_truc=r['kien_truc'], hop=r['kieu_hop'], vong=r['vong'], hat=r['hat'], bien_the=r.get('bien_the', ''),
                        mAP50=r['test']['mAP50'], mAP50_95=r['test']['mAP50_95'],
                        # mAP50 3 lớp: cột mAP50 gốc chia 4 lớp ở vòng 0-2 (MOC=0 hoặc rò rỉ) và 3 lớp ở vòng 3-4, không so được giữa vòng
                        mAP50_3lop=round(sum(r['test']['AP50_theo_lop'][:3]) / 3, 4),
                        R_0_25_Single=r['conf_0_25']['R_theo_lop'][0], R_0_25_Multi=r['conf_0_25']['R_theo_lop'][1],
                        R_0_25_Bypass=r['conf_0_25']['R_theo_lop'][2],
                        AP50_Single=r['test']['AP50_theo_lop'][0], AP50_Multi=r['test']['AP50_theo_lop'][1],
                        AP50_Bypass=r['test']['AP50_theo_lop'][2],
                        AP50_MOC=r['test']['AP50_theo_lop'][3],   # None ở vòng 3, 4 — đúng, không chấm riêng
                        phut=r['phut'], epoch=r['epoch_chay']) for r in R])
print('--- Bảng A: vòng 0, trung bình ± lệch theo hạt ---')
a = df[df.vong == 0].groupby(['kien_truc', 'hop', 'bien_the']).agg(n_hat=('hat', 'count'), mAP50_3lop=('mAP50_3lop', 'mean'), sd=('mAP50_3lop', 'std'), epoch=('epoch', 'mean'),
                                                       mAP50_95=('mAP50_95', 'mean'), AP50_Single=('AP50_Single', 'mean'),
                                                       AP50_Multi=('AP50_Multi', 'mean'), AP50_Bypass=('AP50_Bypass', 'mean'),
                                                       phut=('phut', 'mean')).round(4)
print(a.to_string())
print()
print('--- B3: yolo26s-obb, theo vòng (hạt 0) ---')
b = df[(df.kien_truc == 'yolo26s-obb') & (df.hat == 0) & (df.bien_the == '')].sort_values('vong')[['vong', 'mAP50_3lop', 'mAP50', 'mAP50_95', 'AP50_Single', 'AP50_Multi', 'AP50_Bypass', 'R_0_25_Single', 'R_0_25_Multi', 'R_0_25_Bypass']]
print(b.to_string(index=False))
if len(b) == 5: print('\ntrung bình 5 vòng: mAP50 3 lớp %.4f ± %.4f' % (b.mAP50_3lop.mean(), b.mAP50_3lop.std()))
df.to_csv(f'{KQ}/tong_hop.csv', index=False); print('\n->', f'{KQ}/tong_hop.csv')